In [58]:
import torch
import torch.nn as nn
import numpy as np
import math

# **Encoder**

In [59]:
class InputEmbeddings(nn.Module):
    def __init__(self,d_model:int,vocab_size:int):
        super().__init__()
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.embedding=nn.Embedding(vocab_size,d_model)
    def forward(self,x):
        # 这里乘了d_model
        x=self.embedding(x)*math.sqrt(self.d_model)
        return x

In [60]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model:int,seq_len:int,dropout:float):
        super().__init__()
        self.d_model=d_model
        self.seq_len=seq_len
        self.dropout=nn.Dropout(dropout)
        # (seq_len,d_model)
        pe=torch.zeros(seq_len,d_model)
        # (seq_len)
        position=torch.arange(0,seq_len,dtype=torch.float).unsqueeze(1)
        div_term=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(100000.0)/d_model))
        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        # (1,seq_len,d_model)
        pe=pe.unsqueeze(0)
        # 防止被optimizer更新
        self.register_buffer('pe',pe)
    def forward(self,x):
        x=x+(self.pe[:,:x.shape[1],:])
        return self.dropout(x)

In [61]:
class LayerNormalization(nn.Module):
    def __init__(self,eps:float=1e-6):
        super().__init__()
        self.eps=eps
        self.alpha=nn.Parameter(torch.ones(1))
        self.bias=nn.Parameter(torch.zeros(1))
    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True)
        std=x.std(dim=-1,keepdim=True)
        return self.alpha*(x-mean)/(std+self.eps)+self.bias

In [62]:
class FeedForwardBlock(nn.Module):
    def __init__(self,d_model,d_ff,dropout:float):
        super().__init__()
        self.d_model=d_model
        self.d_ff=d_ff
        self.dropout=nn.Dropout(dropout)
        self.linear1=nn.Linear(d_model,d_ff)
        self.linear2=nn.Linear(d_ff,d_model)
    def forward(self,x):
        x=self.linear2(self.dropout(torch.relu(self.linear1(x))))
        return x

In [63]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self,d_model:int,h:int,dropout:float):
        super().__init__()
        #超参数
        self.d_model=d_model
        self.h=h
        assert d_model%self.h==0
        self.d_k=d_model//self.h
        #权重矩阵
        self.w_q=nn.Linear(d_model,d_model)
        self.w_k=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_o=nn.Linear(d_model,d_model)
        #Dropout层
        self.dropout=nn.Dropout(dropout)
    @staticmethod
    def attention(query,key,value,mask,dropout:nn.Dropout):
        d_k=query.shape[-1]
        attention_scores=(query@key.transpose(-2,-1))/math.sqrt(d_k)
        # casual mask
        if mask is not None:
            attention_scores=attention_scores.masked_fill(mask==0,-1e9)
        attention_scores=attention_scores.softmax(dim=-1)
        if dropout is not None:
            attention_scores=dropout(attention_scores)
        return attention_scores@value
    #input
    def forward(self,q,k,v,mask):
        query=self.w_q(q)
        key=self.w_k(k)
        value=self.w_v(v)
        #every head can see full attention context
        query=query.view(query.shape[0],query.shape[1],self.h,self.d_k).transpose(1,2)
        key=key.view(key.shape[0],key.shape[1],self.h,self.d_k).transpose(1,2)
        value=value.view(value.shape[0],value.shape[1],self.h,self.d_k).transpose(1,2)
        x=MultiHeadAttentionBlock.attention(query,key,value,mask,self.dropout)
        # shape:(batch,head,seq_len,d_k)
        x=x.transpose(1,2).contiguous().view(x.shape[0],-1,self.h*self.d_k)
        x=self.w_o(x)
        return x

In [64]:
class ResidualConnection(nn.Module):
    def __init__(self,dropout:float):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
        self.norm=LayerNormalization()
    def forward(self,x,sublayer):
        #Pre-Norm
        return x+self.dropout(sublayer(self.norm(x)))
        

In [65]:
class EncoderBlock(nn.Module):
    def __init__(self,self_attention_block:MultiHeadAttentionBlock,feed_forward_block:FeedForwardBlock,dropout:float):
        super().__init__()
        self.self_attention_block= self_attention_block
        self.feed_forward_block=feed_forward_block
        self.residual_connections=nn.ModuleList([ResidualConnection(dropout) for i in range(2)])
    def forward(self,x,src_mask):
        x=self.residual_connections[0](x,lambda x:self.self_attention_block(x,x,x,src_mask))
        x=self.residual_connections[1](x,self.feed_forward_block)
        return x
        
        

In [66]:
class Encoder(nn.Module):
    def __init__(self,layers:nn.ModuleList):
        super().__init__()
        self.layers=layers
        self.norm=LayerNormalization()
    def forward(self,x,src_mask):
        for layer in self.layers:
            x=layer(x,src_mask)
        # 这里norm的作用实际上是缓解pre-norm导致的方差很大
        return self.norm(x)

In [67]:
mha=MultiHeadAttentionBlock(d_model=512,h=8,dropout=0.1)
batch_size=2
seq_len=5
d_model=512
x=torch.randn(batch_size,seq_len,d_model)
mask=torch.tril(torch.ones(1,1,seq_len,seq_len))
output=mha(x,x,x,mask)
print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}") 
# 预期输出: torch.Size([2, 5, 512])

assert output.shape == (batch_size, seq_len, d_model), "输出维度不正确！"
print("测试通过！")


输入形状: torch.Size([2, 5, 512])
输出形状: torch.Size([2, 5, 512])
测试通过！


In [68]:
# --- 测试代码 ---
def test_encoder():
    # 超参数
    d_model = 512
    vocab_size = 1000
    seq_len = 20
    h = 8
    dropout = 0.1
    
    # 1. 初始化各层
    embed = InputEmbeddings(d_model, vocab_size)
    pe = PositionalEncoding(d_model, seq_len, dropout)
    
    # 2. 构造 Encoder 模块
    mha = MultiHeadAttentionBlock(d_model, h, dropout)
    ff = FeedForwardBlock(d_model, d_ff=2048, dropout=dropout)
    encoder_block = EncoderBlock(mha, ff, dropout)
    encoder = Encoder(nn.ModuleList([encoder_block for _ in range(3)])) # 3层 Encoder

    # 3. 构造测试数据
    # Batch=2, Seq_len=10
    sample_input = torch.randint(0, vocab_size, (2, 10))
    # 构造 Mask (假设前5个词有效，后5个是 padding)
    mask = torch.ones(1, 1, 10, 10) 

    # 4. 前向传播
    x = embed(sample_input)
    x = pe(x)
    output = encoder(x, mask)

    print(f"输入 ID 形状: {sample_input.shape}")
    print(f"Encoder 输出形状: {output.shape}")
    assert output.shape == (2, 10, 512), "输出维度不匹配！"
    print("✅ Transformer Encoder 测试成功!")

test_encoder()

输入 ID 形状: torch.Size([2, 10])
Encoder 输出形状: torch.Size([2, 10, 512])
✅ Transformer Encoder 测试成功!


# **Decoder**

In [69]:
class DecoderBlock(nn.Module):
    def __init__(self,self_attention_block:MultiHeadAttentionBlock,cross_attention_block:MultiHeadAttentionBlock,feed_forward_block:FeedForwardBlock,dropout:float):
        super().__init__()
        self.self_attention_block=self_attention_block
        self.cross_attention_block=cross_attention_block
        self.feed_forward_block=feed_forward_block
        self.residual_connections=nn.ModuleList([ResidualConnection(dropout) for i in range(3)])
    def forward(self,x,encoder_output,src_mask,tgt_mask):
        x=self.residual_connections[0](x,lambda x:self.self_attention_block(x,x,x,tgt_mask))
        x=self.residual_connections[1](x,lambda x:self.cross_attention_block(x,encoder_output,encoder_output,src_mask))
        x=self.residual_connections[2](x,self.feed_forward_block)
        return x

In [70]:
class Decoder(nn.Module):
    def __init__(self,layers:nn.ModuleList):
        super().__init__()
        self.layers=layers
        self.norm=LayerNormalization()
    def forward(self,x,encoder_output,src_mask,tgt_mask):
        for layer in self.layers:
            x=layer(x,encoder_output,src_mask,tgt_mask)
        return self.norm(x)
        

# **Projection**

In [71]:
class ProjectionLayer(nn.Module):
    def __init__(self,d_model:int,vocab_size:int):
        super().__init__()
        self.proj=nn.Linear(d_model,vocab_size)
    def forward(self,x):
        return torch.log_softmax(self.proj(x),dim=-1)
        

# **Transformer**

In [72]:
class Transformer(nn.Module):
    def __init__(self,encoder:Encoder,decoder:Decoder,src_embed:InputEmbeddings,tgt_embed:InputEmbeddings,src_pos:PositionalEncoding,tgt_pos:PositionalEncoding,projection_layer:ProjectionLayer):
        super().__init__()
        self.encoder=encoder
        self.decoder=decoder
        self.src_embed=src_embed
        self.tgt_embed=tgt_embed
        self.src_pos=src_pos
        self.tgt_pos=tgt_pos
        self.projection_layer=projection_layer
    def encode(self,src,src_mask):
        src=self.src_embed(src)
        src=self.src_pos(src)
        return self.encoder(src,src_mask)
    def decode(self,encoder_output,src_mask,tgt,tgt_mask):
        tgt=self.tgt_embed(tgt)
        tgt=self.tgt_pos(tgt)
        return self.decoder(tgt,encoder_output,src_mask,tgt_mask)
    def project(self,x):
        return self.projection_layer(x)
    def forward(self,src,tgt,src_mask,tgt_mask):
        encode_out=self.encode(src,src_mask)
        decode_out=self.decode(encode_out,src_mask,tgt,tgt_mask)
        return self.project(decode_out)
        

In [73]:
# 对每个class用超参数进行初始化
def build_transformer(src_vocab_size:int,tgt_vocab_size:int,src_seq_len:int,tgt_seq_len:int,d_model:int=512,N:int=6,h:int=8,dropout:float=0.1,d_ff:int=2048)->Transformer:
        #Embedding Layers
        src_embed=InputEmbeddings(d_model,src_vocab_size)
        tgt_embed=InputEmbeddings(d_model,tgt_vocab_size)
        #Positional Embeddings
        src_pos=PositionalEncoding(d_model,src_seq_len,dropout)
        tgt_pos=PositionalEncoding(d_model,tgt_seq_len,dropout)
        # Encoder blocks
        encoder_blocks=[]
        for _ in range(N):
            encoder_self_attention_block=MultiHeadAttentionBlock(d_model,h,dropout)
            feed_forward_block=FeedForwardBlock(d_model,d_ff,dropout)
            encoder_block=EncoderBlock(encoder_self_attention_block,feed_forward_block,dropout)
            encoder_blocks.append(encoder_block)
        # Decoder blocks
        decoder_blocks=[]
        for _ in range(N):
            decoder_self_attention_block=MultiHeadAttentionBlock(d_model,h,dropout)
            decoder_cross_attention_block=MultiHeadAttentionBlock(d_model,h,dropout)
            feed_forward_block=FeedForwardBlock(d_model,d_ff,dropout)
            decoder_block=DecoderBlock(decoder_self_attention_block,decoder_cross_attention_block,feed_forward_block,dropout)
            decoder_blocks.append(decoder_block)
        # Encoder
        encoder=Encoder(nn.ModuleList(encoder_blocks))
        # Decoder
        decoder=Decoder(nn.ModuleList(decoder_blocks))
        # Projection Layer
        projection_layer=ProjectionLayer(d_model,tgt_vocab_size)
        # Transformer
        transformer=Transformer(encoder,decoder,src_embed,tgt_embed,src_pos,tgt_pos,projection_layer)
        for p in transformer.parameters():
            if p.dim()>1:
                nn.init.xavier_uniform_(p)
        return transformer
        

In [74]:
def test_full_transformer():
    # 参数设置
    params = {
        "src_vocab_size": 5000,
        "tgt_vocab_size": 5000,
        "src_seq_len": 100,
        "tgt_seq_len": 100,
        "d_model": 512,
        "N": 2, # 测试用 2 层即可
        "h": 8
    }
    
    model = build_transformer(**params)
    
    # 模拟输入数据 (Batch=2, Seq_len)
    src = torch.randint(0, params["src_vocab_size"], (2, 8))
    tgt = torch.randint(0, params["tgt_vocab_size"], (2, 12))
    
    # 模拟 Mask
    src_mask = torch.ones(1, 1, 1, 8) # 简单全 1 mask
    tgt_mask = torch.tril(torch.ones(1, 1, 12, 12)) # 解码器的因果 mask
    
    # 执行前向传播
    output = model(src, tgt,src_mask, tgt_mask)
    
    print(f"Transformer 运行成功!")
    print(f"输入源形状: {src.shape}")
    print(f"输入目标形状: {tgt.shape}")
    print(f"输出概率分布形状: {output.shape}") 
    # 预期: [2, 12, 5000] (Batch, Tgt_Seq_Len, Tgt_Vocab_Size)

    assert output.shape == (2, 12, params["tgt_vocab_size"])

test_full_transformer()

Transformer 运行成功!
输入源形状: torch.Size([2, 8])
输入目标形状: torch.Size([2, 12])
输出概率分布形状: torch.Size([2, 12, 5000])
